# Lab - Your First Tool Call, End to End

Three tasks that follow one tool call from start to finish. A support assistant that
cannot look at an order can only talk about policy - a tool lets it ask your code.

| Task | What you learn |
| --- | --- |
| 1 | A tool definition is a description: a name, when to use it, and an input schema |
| 2 | Claude only asks - your code runs the function and sends a tool_result back |
| 3 | Send the conversation again, and only now does Claude answer the customer |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Answer in one or two sentences."
)

# A tiny order database: one order, a jacket, delivered twenty days ago.
ORDERS = {
    "ORD-12345678": {
        "item": "jacket",
        "status": "delivered",
        "delivered_days_ago": 20,
    },
}


# The function behind the tool is plain Python: the record, or an error.
def lookup_order(order_id):
    return ORDERS.get(order_id, {"error": "order not found"})

---

## Task 1 - Describe the tool and send it

In [ ]:
# ============================================================
# TASK 1 - Describe the tool and send it
# ============================================================
#
# WHAT TO DO
#   Finish the tool definition, then send the customer's
#   question together with it, and look at the stop_reason.
#
# WHY IT MATTERS
#   The tool definition is a description, not code. It has a
#   name, a description that tells Claude when to use it, and
#   an input schema: the fields the tool needs. Claude reads it
#   like documentation. When the question needs the order,
#   Claude does not answer - it stops with stop_reason
#   "tool_use" and asks for the tool. Nothing has run yet.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The tool definition is a description, not code".
#
# HOW TO DO IT
#   Three blanks:
#
#       "lookup_order"   the tool's name - the name Claude asks for
#       "order_id"       the one field the tool needs
#       TOOLS            the tools list, passed to the request
# ============================================================

messages = [{
    "role": "user",
    "content": "Can I still return the jacket "
               "from order ORD-12345678?",
}]

# BEGIN SOLUTION
TOOLS = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its ID. Returns the "
                       "item, the status and how many days ago "
                       "it was delivered.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID, like ORD-12345678",
                }
            },
            "required": ["order_id"],
        },
    }
]

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    output_config={"effort": "low"},
    system=SYSTEM,
    tools=TOOLS,
    messages=messages,
)
# SCAFFOLD: TOOLS = [
# SCAFFOLD:     {
# SCAFFOLD:         "name": ...,   # "lookup_order" - the name Claude asks for
# SCAFFOLD:         "description": "Look up an order by its ID. Returns the "
# SCAFFOLD:                        "item, the status and how many days ago "
# SCAFFOLD:                        "it was delivered.",
# SCAFFOLD:         "input_schema": {
# SCAFFOLD:             "type": "object",
# SCAFFOLD:             "properties": {
# SCAFFOLD:                 "order_id": {
# SCAFFOLD:                     "type": "string",
# SCAFFOLD:                     "description": "The order ID, like ORD-12345678",
# SCAFFOLD:                 }
# SCAFFOLD:             },
# SCAFFOLD:             "required": [...],   # "order_id" - the one field the tool needs
# SCAFFOLD:         },
# SCAFFOLD:     }
# SCAFFOLD: ]
# SCAFFOLD:
# SCAFFOLD: response = client.messages.create(
# SCAFFOLD:     model=MODEL,
# SCAFFOLD:     max_tokens=300,
# SCAFFOLD:     output_config={"effort": "low"},
# SCAFFOLD:     system=SYSTEM,
# SCAFFOLD:     tools=...,   # TOOLS - without it Claude does not know the tool exists
# SCAFFOLD:     messages=messages,
# SCAFFOLD: )
# END SOLUTION: replace each ... below with the value named beside it

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print("Claude asks for:", block.name, block.input)

check("define_tool", tools=TOOLS, response=response)

---

## Task 2 - Run the function Claude asked for

In [ ]:
# ============================================================
# TASK 2 - Run the function Claude asked for
# ============================================================
#
# WHAT TO DO
#   Find the tool_use block in Claude's reply, call your
#   function with the input Claude chose, and add two turns to
#   the conversation.
#
# WHY IT MATTERS
#   Claude never runs a tool. It asks, your code runs it, and
#   your code sends the result back. Two messages go into the
#   conversation: Claude's own reply, exactly as it came, and a
#   tool_result. The tool_result carries the id of the tool_use
#   block - that is how Claude knows which request it answers -
#   and the result as JSON.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now our code does its part".
#
# HOW TO DO IT
#   Three blanks:
#
#       "tool_use"         the type of block that asks for a tool
#       response.content   Claude's reply, exactly as it came
#       block.id           the id of the tool_use block
# ============================================================

messages = messages[:1]   # provided: back to the question, so re-running is safe
result = None             # provided: stays None if no tool_use block is found

# BEGIN SOLUTION
for block in response.content:
    if block.type == "tool_use":
        print("Claude asks for:", block.name, block.input)
        result = lookup_order(**block.input)
        print("our code returns:", result)
        messages.append(
            {"role": "assistant", "content": response.content}
        )
        messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": block.id,
             "content": json.dumps(result)}
        ]})
# SCAFFOLD: for block in response.content:
# SCAFFOLD:     if block.type == ...:   # "tool_use" - the block that asks for a tool
# SCAFFOLD:         print("Claude asks for:", block.name, block.input)
# SCAFFOLD:         result = lookup_order(**block.input)
# SCAFFOLD:         print("our code returns:", result)
# SCAFFOLD:         messages.append(
# SCAFFOLD:             {"role": "assistant", "content": ...}   # response.content - exactly as it came
# SCAFFOLD:         )
# SCAFFOLD:         messages.append({"role": "user", "content": [
# SCAFFOLD:             {"type": "tool_result", "tool_use_id": ...,   # block.id
# SCAFFOLD:              "content": json.dumps(result)}
# SCAFFOLD:         ]})
# END SOLUTION: replace each ... below with the value named beside it

if result is None:   # provided: no block matched, said plainly
    raise ValueError('No tool_use block was handled. The first blank is "tool_use" - '
                     "the type of the block that asks for a tool.")

check("run_tool", result=result, messages=messages)

---

## Task 3 - Send the result back for the answer

In [ ]:
# ============================================================
# TASK 3 - Send the result back for the answer
# ============================================================
#
# WHAT TO DO
#   Send the conversation again - the question, Claude's
#   request and your tool result - and print the answer.
#
# WHY IT MATTERS
#   Claude now has the order in front of it. This time the
#   stop_reason is "end_turn" and the answer uses the real
#   data: delivered twenty days ago, ten days left. The tools
#   go with this request too: the conversation holds a tool
#   call, and the API rejects it without them.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Finally, we send the conversation again".
#
# HOW TO DO IT
#   Three blanks:
#
#       TOOLS      the same tools as before
#       messages   the list that now holds your tool_result
#       "text"     the type of block that holds the answer
# ============================================================

# BEGIN SOLUTION
final = client.messages.create(
    model=MODEL,
    max_tokens=300,
    output_config={"effort": "low"},
    system=SYSTEM,
    tools=TOOLS,
    messages=messages,
)
print("stop_reason:", final.stop_reason)
for block in final.content:
    if block.type == "text":
        print(block.text)
# SCAFFOLD: final = client.messages.create(
# SCAFFOLD:     model=MODEL,
# SCAFFOLD:     max_tokens=300,
# SCAFFOLD:     output_config={"effort": "low"},
# SCAFFOLD:     system=SYSTEM,
# SCAFFOLD:     tools=...,      # TOOLS - the same tools as before
# SCAFFOLD:     messages=...,   # messages - the question, the request and your result
# SCAFFOLD: )
# SCAFFOLD: print("stop_reason:", final.stop_reason)
# SCAFFOLD: for block in final.content:
# SCAFFOLD:     if block.type == ...:   # "text" - the block that holds the answer
# SCAFFOLD:         print(block.text)
# END SOLUTION: replace each ... below with the value named beside it

check("round_trip", final=final)

---

## Done

A tool call is a round trip. Claude asks with a `tool_use` block, your code runs the
function and answers with a `tool_result` that carries the block's id, and only then does
Claude answer the customer. Next, the same round trip runs in a loop - as many times as
the task needs.